In [3]:
import pandas as pd
import os
# from rapidfuzz import process, fuzz
import unicodedata
import requests
import csv

### Util functions


In [4]:
def save_fpl_players(fpl_subfolders, season):
    print(fpl_subfolders)
    gw_data = pd.read_csv(f'./data/vaastav/data/20{season}/gws/gw1.csv')
    for folder in fpl_subfolders:
        # print(folder)
        player_name = folder.split('/')[6].split('_')
        clean_player_name = " ".join(player_name[0:-1])
        # print(clean_player_name)
        player = gw_data[gw_data['name'] == clean_player_name]
        if player.empty:
            # print(f"Player {clean_player_name} not found in gw_data")
            continue
        # print(player)
        player_df = pd.read_csv(str(folder+'/gw.csv'))

        # fpl_id to player_df
        fpl_id = folder.split('/')[6].split('_')[-1]
        player_df['fpl_id'] = fpl_id
        player_df['position'] = player['position']
        player_df['team'] = player['team']

        player_dir = './data/joint/'+str(season)+'/fpl/'

        # Check if player_dir exists
        if not os.path.exists(player_dir):
            os.makedirs(player_dir)
        player_df.to_csv(player_dir + str(clean_player_name) + '.csv', index_label=False)

    print('successfully cleaned 20'+ season +' fpl data')

# Understat Files
def save_under_players(understat_files, season):
    for file_ in understat_files:
        player_name = file_.split('/')[6].split('_')
        clean_player_name = " ".join(player_name[0:-1])
        player_df = pd.read_csv(str(file_))
        player_dir = './data/joint/'+str(season)+'/understat/'

        # Check if player_dir exists
        if not os.path.exists(player_dir):
            os.makedirs(player_dir)
        player_df.to_csv(player_dir + str(clean_player_name) + '.csv', index_label=False)

    print('successfully cleaned understat 20'+ season +' data')

def joint_players_info(fpl_player_folder_path, understat_player_folder_path, season):
    fpl_subfolders = [ f.path for f in os.scandir(fpl_player_folder_path) if f.is_dir() ]
    under_files = [ f.path for f in os.scandir(understat_player_folder_path) if f.is_file() ]

    save_fpl_players(fpl_subfolders, season)
    save_under_players(under_files, season)

    print('20'+ season +' fpl and understat data now in `joint` folder')

In [ ]:
def merge_fpl_understat_data(fpl_player_folder_path, understat_player_folder_path, season):
    joint_players_info(fpl_player_folder_path, understat_player_folder_path, season)
    joint_fpl_data_path = "./data/joint/"+ season + "/fpl/"
    joint_understat_path = "./data/joint/"+ season + "/understat/"

    # Get the player positions from the previous seasons
    play_merged = pd.read_csv(f'./data/vaastav/data/20{season}/gws/merged_gw.csv')
    unique_players = play_merged.drop_duplicates(subset=['name'])

    name_to_position_series = unique_players.set_index('name')['position']
    name_position_dict = name_to_position_series.to_dict()

    # understat_files = next(os.walk("./data/joint/"+ season + "/understat/"), (None, None, []))[2]  # [] if no file
    # fpl_files = next(os.walk( "./data/joint/"+ season +"/fpl"), (None, None, []))[2]  # [] if no file
    # fpl_understat_id_name = pd.read_csv('./fpl_understat_id_name.csv')

    # Clear out files that don't have both fpl and understat data
    sns_yr = '_'.join(season.split('-'))  # '2021-22' -> '2021'
    fpl_understat_id_name =pd.read_csv('./data/vaastav/data/id_dict_'+ sns_yr +'.csv') # pd.read_csv('./fpl_understat_id_name.csv')
    fpl_understat_id_name.rename(columns={'FPL_Name': 'fpl_name', 'FPL_ID':'fpl_id' , 'Understat_ID': 'understat_id','Understat_Name': 'understat_name'}, inplace=True)
    fpl_names = fpl_understat_id_name['fpl_name'].values.tolist()
    under_names = fpl_understat_id_name['understat_name'].values.tolist()

    understat_files = next(os.walk("./data/joint/"+ season + "/understat/"), (None, None, []))[2]  # [] if no file
    fpl_files = next(os.walk( "./data/joint/"+ season +"/fpl"), (None, None, []))[2]  # [] if no file

    fpl_files_ = [f.split('.')[0] for f in fpl_files]
    understat_files_ = [f.split('.')[0] for f in understat_files]


    # drop fple_files not in fpl_names
    fpl_file_names = [file_ for file_ in fpl_files_ if file_ in fpl_names]
    under_file_name = [file_ for file_ in understat_files_ if file_ in under_names]

    # put back to fpl_files
    fpl_files =[name+'.csv' for name in fpl_file_names]
    understat_files = [name+'.csv' for name in under_file_name]


    understat_names = [file_.split('.')[0] for file_ in understat_files]
    fpl_file_names = [file_.split('.')[0] for file_ in fpl_files]

    for name in fpl_file_names:
        fpl_player = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == name]
        # fpl_player['fpl_name'].values

        if fpl_player['fpl_name'].values.size > 0:
            fpl_player_name = fpl_player['fpl_name'].values[0]
            understat_player_name = fpl_player['understat_name'].values[0]
            # print(fpl_player_name)

            try:
                fpl_player_data = pd.read_csv(joint_fpl_data_path + fpl_player_name+ '.csv')
                understat_player_data = pd.read_csv(joint_understat_path + understat_player_name + '.csv')

            except pd.errors.EmptyDataError:
                print(f"⚠️ Warning: The file '{joint_fpl_data_path + fpl_player_name+ '.csv'}' is empty. Skipping.")
                # You can choose to create an empty DataFrame or just continue
                print(f"Error reading files for {fpl_player_name} or {understat_player_name}. Skipping...")
                continue
            # print(joint_fpl_data_path + fpl_player_name+ '.csv')
            # Change 'kickoff_time' column name to 'date
            fpl_player_data = fpl_player_data.rename(columns={'kickoff_time': 'date'})
            # change the formats: From 2021-10-03T13:00:00Z to 2021-10-03
            fpl_player_data.date = fpl_player_data.date.apply(lambda x: x.split('T')[0])

            # Dates are of the form 2021-10-03T13:00:00Z
            fpl_dates_min = fpl_player_data['date'].min()
            fpl_dates_max = fpl_player_data['date'].max()

            # Filter out player info not in the range of dates we are dealing with
            understat_filtered = understat_player_data[(pd.to_datetime(understat_player_data['date']) >= pd.to_datetime(fpl_dates_min))
                                                        & (pd.to_datetime(understat_player_data['date']) <= pd.to_datetime(fpl_dates_max) )]

            # Marge fpl_player_data with understat_player_data if the dates match
            player_data_merged = fpl_player_data.merge(understat_filtered, on="date")
            # print(player_data_merged.shape)
            # Add player team
            def set_player_team(row):
                # add fpl_id
                row['fpl_id'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_id'].values[0]
                row['understat_id'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['understat_id'].values[0]
                # # add player_team
                if 'team' not in fpl_understat_id_name.columns:
                    # get the player team from understat data
                    # under_player = pd.read_csv(f'./vaastav/data/{season}/understat/Aaron_Cresswell_534.csv')
                    # fpl_player = pd.read_csv(f'./vaastav/data/{season}/players/Aaron_Cresswell_517/gw.csv')
                    # teams = pd.read_csv(f'./vaastav/data/{season}/teams.csv')

                    # Get player team by comparing opponent_team in fpl gw data with teams id in teams data
                    h_team = player_data_merged.iloc[0]['h_team']
                    a_team = player_data_merged.iloc[0]['a_team']
                    # opponent_team = fpl_player.iloc[0]['opponent_team']
                    was_home = player_data_merged.iloc[0]['was_home']
                    row['player_team'] = h_team if was_home else a_team
                else:
                    row['player_team'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['team'].values[0]

                # add FPL_Name
                row['FPL_Name'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_name'].values[0]
                # add Understat_Name
                row['Understat_Name'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['understat_name'].values[0]

                # add fpl position
                if 'fpl_position' not in fpl_understat_id_name.columns:
                   row['position'] = name_position_dict[fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_name'].values[0]]
                else:
                    row['position'] = fpl_understat_id_name[fpl_understat_id_name['fpl_name'] == fpl_player_name]['fpl_position'].values[0]
                return row
            player_data_merged = player_data_merged.apply(set_player_team, axis=1)

            #     return row['h_team'] if row['was_home'] else row['a_team']

            # player_data_merged['player_team'] = player_data_merged.apply(set_player_team, axis=1)


            if(player_data_merged.shape[0]):
                merged_dir = './data/joint/'+ season +'/merged/'
                if not os.path.exists(merged_dir):
                    os.makedirs(merged_dir)

                player_data_merged.to_csv(merged_dir+ fpl_player_name +'.csv', index_label=False )

In [6]:
def add_difficulty(season):
    print('====> Starting to add difficulty features to 20'+season)
    merged = './data/joint/' + season + '/merged/'

    player_names = next(os.walk((merged), (None, None, [])))[2]
    fixtures = pd.read_csv('./data/vaastav/data/20' + season + '/fixtures.csv')

    # Loop over each player file in player_names
    for name in player_names:
        # Load player data
        player = pd.read_csv('./data/joint/' + season + '/merged/' + name)

        # Function to get the difficulty and was_home columns based on the fixture
        def get_fixture_info(row):
            # Filter the relevant fixture
            fixture = fixtures[fixtures['id'] == row['fixture']]
            if not fixture.empty:
                fixture = fixture.iloc[0]  # Get the first (and only) match

                # Get the team difficulties
                team_h_difficulty = fixture['team_h_difficulty']
                team_a_difficulty = fixture['team_a_difficulty']
                event = fixture['event']

                return pd.Series([team_h_difficulty, team_a_difficulty, event])
            else:
                # Return NaN if no matching fixture found
                return pd.Series([None, None, None])

        # Apply the function to each row of player
        player[['team_h_difficulty', 'team_a_difficulty', 'event']] = player.apply(get_fixture_info, axis=1)
        # Update the value to prices
        # player['value'] = player['value']/10

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/' + season + '/merged_extras/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + name, index=False)

In [7]:
def add_xP(season, gwk=None):
    print('================> starting to add xp for season 20'+season)
    if gwk:
        # Define the API endpoint
        url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

        # Send a GET request to the API
        response = requests.get(url)
        player_data = None
        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()  # Parse the JSON data
            players = data['elements']  # Extract the list of players

            player_data = [
                {"id": player["id"],  "name": f"{player['first_name']} {player['second_name']}", "player_team": player["team"], "price": player["now_cost"] / 10, "position": player["element_type"]}
                for player in players
            ]

        else:
            print('Failed to retrieve data')

        fpl_players = pd.DataFrame(player_data)

    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras', [None], [None],[]))[2]
    # players_paths
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras/'+ path)
        merged = pd.read_csv('./data/vaastav/data/20'+ season +'/gws/merged_gw.csv', low_memory=False)

        player = player.drop(['position'], axis=1)
        merged_player = pd.merge(player, merged[['element', 'fixture', 'xP','position']], on=['element', 'fixture'], how='left')
        # drop the duplicates
        merged_player = merged_player.drop_duplicates(subset=['date'])
        # print(path ,player.shape, merged.shape, merged_player.shape)

        if gwk:
            merged_player = merged_player.reindex(merged_player.index.tolist()  + list([merged_player.index[-1]+1]))


            player_id = int(merged_player.iloc[-2, merged_player.columns.get_loc('element')])
            fpl_data = fpl_players[fpl_players['id'] == player_id]

            merged_player.iloc[-1, merged_player.columns.get_loc('event')] = gwk
            merged_player.iloc[-1, merged_player.columns.get_loc('value')] = fpl_data['price'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('position')] = fpl_data['position'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('fpl_id')] = player['element'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('understat_id')] = player['understat_id'].values[0]
            merged_player.iloc[-1, merged_player.columns.get_loc('player_team')] = fpl_data['player_team'].values[0]


        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/'+ season +'/merged_extras_xP/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        merged_player.to_csv(new_col_dir + path, index=False)

In [8]:
def add_rolling_avgs(season):
    print('================> starting to roll by 2 for season 20'+season)
    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras_xP', [None], [None],[]))[2]
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras_xP/'+ path,sep=',', skipinitialspace=True)

        features = ['clean_sheets', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'goals_conceded', 'goals_scored', 'ict_index',
                        'influence', 'creativity', 'threat', 'minutes', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'yellow_cards', 'saves', 'starts',
                        'team_a_score', 'team_h_score', 'total_points', 'goals', 'shots', 'xG', 'xA', 'assists_x', 'assists_y', 'key_passes', 'npg', 'npxG', 'xGChain',  'xGBuildup',  'xP']

        for i in [1,3,5]:
            # Compute rolling 5-game averages, shifted by 1 (efficient method)
            # rolling_means = round(player[features].rolling(window=i).mean().shift(1), 2)

            # Compute rolling 5-game sum, shifted by 1
            rolling_sum = player[features].rolling(window=i, min_periods=1).sum().shift(1)

            # Divide the sum by the fixed window size 'i'
            rolling_means = round(rolling_sum / i, 2)
            rolling_means.columns = [f"{col}_{i}" for col in rolling_means.columns]  # Rename new columns
            # print(i)
            # Concatenate new rolling mean columns efficiently
            player = pd.concat([player, round(rolling_means, 2)], axis=1)
            # print(f"Rolling {i} added for {path}", player)
            # Defragment memory
            player = player.copy()

        # divide value by 10 to get player price
        player['value'] = player['value'] / 10
        # Save the updated DataFrame with the new columns
        # print(player[['minutes', 'minutes_3','minutes_4', 'minutes_5']].head(10))
        new_col_dir = f'./data/joint/{season}/merged_extras_rolled/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + path, index=False)

    print('<<<<================ Done rolling season 20'+season)

In [9]:
def owenership_change(season, gwk=None):
    print('================> starting to add ownership change for season 20'+season)

    if gwk:
        # Define the API endpoint
        url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

        # Send a GET request to the API
        response = requests.get(url)
        player_data = None
        # Check if the request was successful
        if response.status_code == 200:
            data = response.json()  # Parse the JSON data
            players = data['elements']  # Extract the list of players
            # print(players[0]['transfers_in_event'], players[0]['transfers_out_event'])
            # transfers_in = players['transfers_in_event']
            # transfers_out = players['transfers_out_event']

            transfers = [{"id": player["id"], "transfers_in": player["transfers_in_event"], "transfers_out": player["transfers_out_event"]} for player in players]
        else:
            print('Failed to retrieve data')

        transfers_df = pd.DataFrame(transfers)

    players_paths = next(os.walk('./data/joint/'+ season +'/merged_extras_rolled', [None], [None],[]))[2]
    for path in players_paths:
        player = pd.read_csv('./data/joint/'+ season +'/merged_extras_rolled/'+ path,sep=',', skipinitialspace=True)
        player['ownership_change'] = player['selected'].diff().fillna(0)

        def ownership_change(row):
            net_transfers = row['transfers_in'] - row['transfers_out']
            total_transfers = row['transfers_in'] + row['transfers_out']
            net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

            return net_transfers_pct

        player['percenatge_net_transfers'] = player.apply(ownership_change, axis=1)

        if gwk:
            player_id = int(player.iloc[-2, player.columns.get_loc('element')])
            transfer_data = transfers_df[transfers_df['id'] == player_id]
            net_transfers = transfer_data['transfers_in'].values[0] - transfer_data['transfers_out'].values[0]
            total_transfers = transfer_data['transfers_in'].values[0] + transfer_data['transfers_out'].values[0]
            net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

            player.iloc[-1, player.columns.get_loc('percenatge_net_transfers')] = net_transfers_pct

        # Save the updated DataFrame with the new columns
        new_col_dir = f'./data/joint/{season}/merged_extras_rolled_net_transfers/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        player.to_csv(new_col_dir + path, index=False)

In [10]:
def odds(sns, nxt_gw=0):
    print('================> starting adding odds for sns 20'+sns)

    teams_25_26 = [
        {"name":"Arsenal", "id":"3","shortName":"Arsenal","abbr":"ARS"},
        {"name":"Aston Villa", "id":"7","shortName":"Aston Villa","abbr":"AVL"},
        {"name":"Bournemouth", "id":"91","shortName":"Bournemouth","abbr":"BOU"},
        {"name":"Brentford", "id":"94","shortName":"Brentford","abbr":"BRE"},
        {"name" :"Brighton and Hove Albion","id":"36","shortName":"Brighton","abbr":"BHA"},
        {"name":"Burnley","id":"90","shortName":"Burnley","abbr":"BUR"},
        {"name":"Chelsea","id":"8","shortName":"Chelsea","abbr":"CHE"},
        {"name":"Crystal Palace","id":"31","shortName":"Crystal Palace","abbr":"CRY"},
        {"name":"Everton","id":"11","shortName":"Everton","abbr":"EVE"},
        {"name":"Fulham","id":"54","shortName":"Fulham","abbr":"FUL"},
        {"name":"Leeds United","id":"2","shortName":"Leeds","abbr":"LEE"},
        {"name":"Liverpool","id":"14","shortName":"Liverpool","abbr":"LIV"},
        {"name":"Manchester City","id":"43","shortName":"Man City","abbr":"MCI"},
        {"name":"Manchester United","id":"1","shortName":"Man Utd","abbr":"MUN"},
        {"name":"Newcastle United","id":"4","shortName":"Newcastle","abbr":"NEW"},
        {"name":"Nottingham Forest","id":"17","shortName":"Nott'm Forest","abbr":"NFO"},
        {"name":"Sunderland","id":"56","shortName":"Sunderland","abbr":"SUN"},
        {"name":"Tottenham Hotspur","id":"6","shortName":"Spurs","abbr":"TOT"},
        {"name":"Tottenham Hotspur","id":"6","shortName":"Tottenham","abbr":"TOT"},
        {"name":"Tottenham Hotspur","id":"6","shortName":"Tottenham Hotspur","abbr":"TOT"},
        {"name":"West Ham United","id":"21","shortName":"West Ham","abbr":"WHU"},
        {"name":"Wolverhampton Wanderers","id":"39","shortName":"Wolves","abbr":"WOL"}]

    # create lookup dictionary
    team_name_to_id_25_26 = {team['shortName']: team['name'] for team in teams_25_26}

    # Load the data data once to avoid redundant file reads
    data = pd.read_csv('./data/odds/E0 '+ sns +'.csv')

    data = data.rename(columns={'HomeTeam': 'h_team', 'AwayTeam': 'a_team'})

    # Update the odds-data names of the team to match the fpl names
    # use teams dict to update team names
    def update_name(row):
        if row['h_team'] in team_name_to_id_25_26:
            row['h_team'] = team_name_to_id_25_26[row['h_team']]
        if row['a_team'] in team_name_to_id_25_26:
            row['a_team'] = team_name_to_id_25_26[row['a_team']]
        return row

    data = data.apply(update_name, axis=1)

    players_paths = next(os.walk('./data/joint/'+ sns +'/merged_extras_rolled_net_transfers', [None], [None],[]))[2]
    # print(len(players_paths), 'players to add odds to')
    for path in players_paths:
        rolled = pd.read_csv('./data/joint/'+ sns +'/merged_extras_rolled_net_transfers/'+ path,sep=',', skipinitialspace=True)

        def update_player_team(row):
            if row['player_team'] in team_name_to_id_25_26:
                row['player_team'] = team_name_to_id_25_26[row['player_team']]
            if row['opponent_team'] in team_name_to_id_25_26:
                row['opponent_team'] = team_name_to_id_25_26[row['opponent_team']]
            if row['h_team'] in team_name_to_id_25_26:
                row['h_team'] = team_name_to_id_25_26[row['h_team']]
            if row['a_team'] in team_name_to_id_25_26:
                row['a_team'] = team_name_to_id_25_26[row['a_team']]
            return row
        rolled = rolled.apply(update_player_team, axis=1)

        def add_odds(row, data):
            # Filter the data DataFrame for the matching teams
            # print(data)
            match = data[(data['h_team'] == row['h_team']) & (data['a_team'] == row['a_team'])]
            # Check if a match is found
            if not match.empty:
                # print(row['value'])

                # Extract the relevant data values
                # Convert the data to probabilities
                odds_ = match.iloc[0]

                WHH = round(1/odds_['B365H'], 5)
                WHD = round(1/odds_['B365D'], 5)
                WHA = round(1/odds_['B365A'], 5)

                # Normalize the probabilities (to make the probabilities sum to 100%)
                WH_sum = WHH + WHD + WHA
                WHH_ = round(WHH/WH_sum, 3)
                WHD_ = round(WHD/WH_sum, 3)
                WHA_ = round(WHA/WH_sum, 3)

                # if row['minutes'] == 0:
                #     xG_90 = 0
                #     xA_90 = 0
                # else:
                #     xG_90 = (row['xG']/row['minutes'])*90
                #     xA_90 = (row['xA']/row['minutes'])*90
                pts_bps = row['total_points'] - row['bonus']
                return pd.Series([pts_bps,  WHH_, WHD_, WHA_])
            else:
                # Return NaN for rows with no match
                return pd.Series([None,None, None, None])

        # # Apply the function to the 'rolled' DataFrame
        # rolled[['pts_bps', 'whh', 'whd', 'wha']] = rolled.apply(add_odds, axis=1, data=data)

        # 1. Run .apply() and store the results in a NEW DataFrame
        new_columns = rolled.apply(add_odds, axis=1, data=data)

        # Optional but good practice: ensure the new columns have the correct names
        # (Your .apply() function might already return a Series with names, but this is a safe way to be sure)
        new_columns.columns = ['pts_bps', 'whh', 'whd', 'wha']

        # 2. Join the original DataFrame with the new columns all at once
        rolled = pd.concat([rolled, new_columns], axis=1)

        if nxt_gw:
            odds_nxt = pd.read_csv(f'./data/odds/odds_{nxt_gw}.csv')
            odds_nxt = odds_nxt.rename(columns={'HomeTeam': 'h_team', 'AwayTeam': 'a_team'})

            def update_name(row):
                if row['h_team'] in team_name_to_id_25_26:
                    row['h_team'] = team_name_to_id_25_26[row['h_team']]
                if row['a_team'] in team_name_to_id_25_26:
                    row['a_team'] = team_name_to_id_25_26[row['a_team']]
                return row
            odds_nxt = odds_nxt.apply(update_name, axis=1)

            # print(odds_nxt["a_team"].unique(), odds_nxt['h_team'].unique(), rolled['player_team'].unique())
            opp_team = rolled['opponent_team']
            player_team = rolled['player_team'].loc[0]
            # print(row)

            team_odds_nxt = odds_nxt[(odds_nxt['h_team'] == player_team) | (odds_nxt['a_team']==player_team)]
            # print('------------------->', team_odds_nxt)
            if(team_odds_nxt.empty):
                print(f'No odds found for {player_team} in gw{nxt_gw}, skipping...')
                continue


            h_team = team_odds_nxt.loc[:, 'h_team'].values[0]
            a_team = team_odds_nxt.loc[:, 'a_team'].values[0]
            whh = team_odds_nxt.loc[:,'WHH'].values[0]
            whd = team_odds_nxt.loc[:,'WHD'].values[0]
            wha = team_odds_nxt.loc[:,'WHA'].values[0]
            h_fdr = team_odds_nxt.loc[:,'h_fdr'].values[0]
            a_fdr = team_odds_nxt.loc[:,'a_fdr'].values[0]
            was_home = True if h_team == player_team else False

            rolled.iloc[-1, rolled.columns.get_loc('whh')] = whh
            rolled.iloc[-1, rolled.columns.get_loc('whd')] = whd
            rolled.iloc[-1, rolled.columns.get_loc('wha')] = wha
            rolled.iloc[-1, rolled.columns.get_loc('was_home')] = was_home
            rolled.iloc[-1, rolled.columns.get_loc('h_team')] = h_team
            rolled.iloc[-1, rolled.columns.get_loc('a_team')] = a_team
            rolled.iloc[-1, rolled.columns.get_loc('team_h_difficulty')] = h_fdr
            rolled.iloc[-1, rolled.columns.get_loc('team_a_difficulty')] = a_fdr

        # Save the updated DataFrame with the new columns
        new_col_dir = './data/joint/'+ sns +'/merged_extras_odds/'
        if not os.path.exists(new_col_dir):
            os.makedirs(new_col_dir)
        rolled.to_csv(new_col_dir + path, index=False)

In [11]:
def merge_files(season):
    print('starting to merge files for 20'+ season)
    paths = next(os.walk('./data/joint/'+ season +'/merged_extras_odds', [None], [None],[]))[2]
    print(len(paths))
    files_list = [pd.read_csv('./data/joint/'+ season +'/merged_extras_odds/' + path)  for  path in paths ]
    merged_files = pd.concat(files_list)

    # print(merged_files['fpl_id'])
    # Save the new DataFrame
    new_col_dir = './data/joint/'+ season +'/'

    merged_files.to_csv(new_col_dir  +'merged_player_data.csv', index=False)

In [ ]:
# # For the current season
# %run "./getplayerdetails_24_25.ipynb"